# Library Import

In [1]:
import gymnasium as gym
import torch
from torch import nn
import torch.optim as optim
from torch.distributions import Normal      # normal dist

import numpy as np
import matplotlib.pyplot as plt
from statistics import mean, stdev
import random

import re, os, json, time
from datetime import datetime
from collections import deque
from tqdm import tqdm

# --- import the custom-made TD3 algorithm
import sys
sys.path.insert(0,'..')
from algos import TD3

# Class and Parameter Definition

In [ ]:
def set_global_seed(env: gym.Env, seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
    env.action_space.seed(seed)

# Reacher Training

In [2]:
model_registry = {
    'TD3_v0': {
        'actor_config': [128,128,128],
        'critic_config': [128,128,128]
    },
    'TD3_v1': {
        'actor_config': [256, 256],
        'critic_config': [256, 256]
    },
    'TD3_v2': {
        'actor_config': [256, 256, 256],
        'critic_config': [256, 256, 256]
    },
    'TD3_v3': {
        'actor_config': [256, 256, 256, 256],
        'critic_config': [256, 256, 256, 256]
    }
}

MODEL_NAME = 'TD3_v1'
ALPHA1 = 5e-4
ALPHA2 = 5e-4
BETA = 5e-4
GAMMA = 0.99
TAU_C = 5e-3
TAU_A = 5e-3
SIGMA = 0.05
CLIP = 0.1

BUFFER_SIZE = 1_000_000
BUFFER_INIT = 100
BATCH_SIZE = 256

UPDATE_FREQ = 2
UPDATE_STEP = 4
TRAIN_ITER = 100_000
TRAIN_CRIT = {"pass_limit": 1, "pass_score": -4.5, 'coeff_var_limit': 1.0}
RESULT_FOLDER = 'reacher_TD3_results'
CUDA_ENABLED = True
EARLY_STOP = True

In [3]:
env = gym.make("Reacher-v5", render_mode=None)
# env_val = gym.make("Reacher-v5", render_mode=None)


# set_global_seed(env,seed=seed)
# set_global_seed(env_val,seed=seed)
for i in range(1):    
    seed = np.random.randint(1,100)
    TD3_experiment = TD3(model_name = MODEL_NAME, model_registry=model_registry, env=env,
                     alpha1=ALPHA1,alpha2=ALPHA2,beta=BETA,gamma=GAMMA,
                     tau_c=TAU_C,tau_a=TAU_A,sigma=SIGMA,clip=CLIP,
                     buffer_size=BUFFER_SIZE,buffer_init=BUFFER_INIT, batch_size=BATCH_SIZE, 
                     update_f=UPDATE_FREQ, update_step=UPDATE_STEP, iter=TRAIN_ITER,
                     seed=seed,
                     train_crit=TRAIN_CRIT,
                     result_folder=RESULT_FOLDER,
                     cuda_enabled=CUDA_ENABLED)                 
    TD3_experiment.train(early_stop=EARLY_STOP,verbose=False)

run_00097:   4%|█▋                                           | 3797/100000 [00:13<05:43, 280.27it/s]


KeyboardInterrupt: 

In [ ]:
TD3_experiment.reward_hist

# Visualize the results

In [ ]:
def load_model(q_network: nn.Module, model_path):
    checkpoint = torch.load(model_path)
    q_network.load_state_dict(checkpoint['model_state_dict'])

def EMA_filter(reward: list, alpha):
        ''' Function that runs an exponential moving average filter along a datastream '''
        output = np.zeros(len(reward)+1)
        output[0] = reward[0]
        for idx, item in enumerate(reward):
            output[idx+1] = (1 - alpha) * output[idx] + alpha * item
        
        return output

def plot_fn(history, xlabel:str='step', ylabel: str='reward', alpha: float=0.0):
        ''' Function that plots the reward and filtered reward per episode, then saves the plot in a specified save directory'''
        n_episodes= len(history)
        episodes = range(n_episodes)
        filtered_reward_hist = EMA_filter(history, alpha)

        legend = []
        plt.figure(figsize=(20,6))
        plt.plot(episodes, history[:n_episodes], color = "blue"); legend.append(ylabel)
        if alpha:
            plt.plot(episodes, filtered_reward_hist[:n_episodes], color = "red"); legend.append('filtered '+ylabel)
        # plt.title(f'Total reward per episode - {self.hyperparam_config}')
        plt.xlabel(xlabel)
        plt.ylabel(ylabel)
        plt.legend(legend)
        plt.grid(which='both')

        plt.tight_layout()
        # if self.save_path:
        #     plt.savefig(os.path.join(self.save_path,'reward_history.png'))
        plt.show()

In [ ]:
for i in range(5):
    plot_fn(TD3_experiment.reward_hist, alpha=0.1*i)

In [ ]:
np.max(TD3_experiment.reward_hist[1:])

In [ ]:
len(TD3_experiment.reward_hist)